# Querying the flights with MobilitySpark

This notebook runs the queries of the section "Analysis of the Flights" of the chapter
Large-Scale Mobility Databases, which analyze the flights of **Sunday, March 1, 2026** over Europe
before the dashboards are built. It reads the tables that notebook
`8.3 Create tables from scratch-flights-MobilitySpark` builds, and the table `FlightsEU` of the
dashboards (`dashboards/dashboards_tables.sql`), which it builds when it is absent:

| Table | Content |
|---|---|
| `Countries` | The countries of Natural Earth that intersect the envelope of Europe, in EPSG:3035 |
| `MorningFlights` | The flights over Europe between 8 and 9 o'clock |
| `BrusselsDepartures` | The flights leaving Brussels, with their route and their destination |
| `CountryTraffic` | The flights over each country and the distance they fly there |
| `AirborneCount` | The number of aircraft in the air over Europe at each instant of the day |
| `Holding` | The part of each flight within 15 km of a holding fix of London Heathrow |

The countries are the shapefile `ne_50m_admin_0_countries` of
[Natural Earth](https://www.naturalearthdata.com/), which `flights/fetch_data.sh` of this
repository downloads into `/opt/data/flights/countries`, read by the shapefile reader of Apache
Sedona. The script `flights/flights_queries.sql` runs the same queries in MobilityDB, whose
answers the notebook reproduces.

In [ ]:
## Run this cell FIRST

import subprocess
import time
import sys

print("Starting Safe YARN Cluster Cleanup...")

# 1. Clean up local zombie spark references before querying YARN
if 'spark' in sys.modules or 'pyspark' in sys.modules:
    try:
        from pyspark.sql import SparkSession
        active_session = SparkSession.getActiveSession()
        if active_session:
            print("Stopping local legacy session safely...")
            active_session.stop()
    except Exception:
        pass

try:
    # 2. Query YARN for active sessions
    cmd_list = "yarn application -list -appStates RUNNING,ACCEPTED"
    result = subprocess.check_output(cmd_list, shell=True, text=True)
    
    lines = result.strip().split("\n")
    apps_killed = 0
    
    for line in lines:
        if "application_" in line:
            app_id = line.split()[0]
            print(f"Killing conflicting YARN session: {app_id}")
            subprocess.call(f"yarn application -kill {app_id}", shell=True)
            apps_killed += 1
            
    if apps_killed > 0:
        print("Waiting for resource managers to clear memory allocations...")
        time.sleep(6)
        print("Cluster environment successfully normalized!")
    else:
        print("Cluster queue is already wide open and clean.")

except Exception as e:
    print(f"YARN query bypassed: {e}")

In [ ]:
## Run this cell SECOND

import os
from pyspark.sql import SparkSession

print("Allocating new Spark Session...")

# The master is YARN in the cluster of the chapter; SPARK_MASTER=local[*] runs the notebook on
# a single machine. The session time zone is UTC, the time zone of the OpenSky timestamps.
spark = SparkSession.builder \
    .appName("MobilitySpark-Flights") \
    .config("spark.master", os.environ.get("SPARK_MASTER", "yarn")) \
    .config("spark.submit.deployMode", "client") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "2g")) \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

# Register the typed SQL surface of MobilitySpark: each MobilityDB type (tgeogpoint, tfloat,
# ttext, ...) is a Spark type, and each MobilityDB function is called by its MobilityDB name
spark._jvm.org.mobilitydb.spark.sql.MobilitySparkSql.registerAll(spark._jsparkSession)

print("Spark Session successfully attached to " + spark.sparkContext.master)

In [ ]:
## Run this cell THIRD

from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic


@register_cell_magic
def sql(line, cell):

    # Eliminar comentarios SQL de líneas completas
    lines = []
    for line in cell.splitlines():
        if not line.strip().startswith("#"):
            lines.append(line)

    cell = "\n".join(lines)

    # Separar sentencias
    statements = [
        s.strip()
        for s in cell.split(";")
        if s.strip()
    ]

    for statement in statements:
        print("\n==============================")
        print("RUNNING:")
        print(statement)
        print("==============================")

        result = spark.sql(statement)
        # Only attempt to call show() if the query actually returns data rows
        if result.schema:
            result.show(truncate=False)
        else:
            print("Command executed successfully.")
            

print("cell magic %%sql prepared")

In [ ]:
## Run this cell FOURTH

# Register the Delta tables of notebook 8.3, build flightseu with the statements of
# dashboards/dashboards_tables.sql when the warehouse does not hold it, and remove the tables of
# this notebook left by a previous run

from py4j.java_gateway import java_import

java_import(spark._jvm, "org.apache.hadoop.fs.Path")

warehouse = spark.conf.get("spark.sql.warehouse.dir")
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()


def in_warehouse(table):
    path = spark._jvm.Path(warehouse + "/" + table + "/_delta_log")
    return path.getFileSystem(hadoop_conf).exists(path)


for table in ["flights", "flightseu"]:
    if in_warehouse(table):
        spark.sql(f"CREATE TABLE IF NOT EXISTS {table} USING DELTA LOCATION '{warehouse}/{table}'")

if not in_warehouse("flightseu"):
    with open("dashboards/dashboards_tables.sql") as f:
        lines = [ln for ln in f.read().splitlines() if not ln.lstrip().startswith("--")]
    for statement in [s.strip() for s in "\n".join(lines).split(";") if s.strip()][:2]:
        spark.sql(statement)

for table in ["countries", "morningflights", "brusselsdepartures", "countrytraffic",
              "airbornecount", "holding"]:
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    path = spark._jvm.Path(warehouse + "/" + table)
    path.getFileSystem(hadoop_conf).delete(path, True)

print(spark.sql("SELECT count(*) FROM FlightsEU").collect()[0][0], "flights in flightseu")

## The countries

The view `CountriesInput` reads the shapefile with Sedona, in WGS84. The table `Countries` keeps
the countries that intersect the envelope of Europe, their boundary projected to EPSG:3035 by
Sedona and read by MEOS from its HexEWKB.

In [ ]:
%%sql

CREATE OR REPLACE TEMPORARY VIEW CountriesInput USING shapefile
  OPTIONS (path 'file:///opt/data/flights/countries');

DROP TABLE IF EXISTS Countries;

CREATE TABLE Countries USING DELTA AS
SELECT trim(TRAILING chr(0) FROM NAME) AS Name, ADM0_A3 AS Code,
  geoTransform(geometryFromHexEWKB(ST_AsHEXEWKB(ST_SetSRID(geometry, 4326))),
    3035) AS Geom
FROM CountriesInput
WHERE ST_Intersects(geometry,
  ST_PolygonFromEnvelope(-35.58, 24.60, 44.83, 84.73));

SELECT count(*) FROM Countries;

## Query 1: the flights over Europe between 8 and 9 o'clock

Each flight of `FlightsEU` is restricted to the hour with `atTime` and to the envelope of Europe
in EPSG:3035 with `atGeometry`. The length of a restriction is given by `geoLength`, the function
`length` of MobilityDB, whose name Spark owns.

In [ ]:
%%sql

DROP TABLE IF EXISTS MorningFlights;

CREATE TABLE MorningFlights USING DELTA AS
WITH Europe(Env) AS (
  SELECT geoTransform(segmentize(geometryFromHexEWKB(ST_AsHEXEWKB(ST_SetSRID(
    ST_PolygonFromEnvelope(-35.58, 24.60, 44.83, 84.73), 4326))), 1), 3035) ),
TimePeriod(Period) AS (
  SELECT tstzspanFromText('[2026-03-01 08:00:00+00, 2026-03-01 09:00:00+00)') )
SELECT ICAO24, CallSign, atGeometry(atTime(Flight, Period), Env) AS Trip
FROM FlightsEU, Europe, TimePeriod
WHERE atGeometry(atTime(Flight, Period), Env) IS NOT NULL;

SELECT count(*) AS Flights, count(DISTINCT ICAO24) AS Aircraft,
  sum(geoLength(Trip)) / 1000 AS Km
FROM MorningFlights;

## Query 2: the flights leaving Brussels

A flight leaves Brussels when its first position, `startValue`, lies in the envelope of Brussels;
its destination is its last position, `endValue`.

In [ ]:
%%sql

DROP TABLE IF EXISTS BrusselsDepartures;

CREATE TABLE BrusselsDepartures USING DELTA AS
WITH Brussels(Env) AS (
  SELECT geographyFromHexEWKB(ST_AsHEXEWKB(ST_SetSRID(
    ST_PolygonFromEnvelope(4.25, 50.75, 4.60, 50.95), 4326))) )
SELECT ICAO24, CallSign, trajectory(Flight) AS Route, endValue(Flight) AS Destination,
  geoLength(Flight) / 1000 AS Km
FROM Flights, Brussels
WHERE geoIntersects(startValue(Flight), Env);

SELECT count(*) AS Flights, count(DISTINCT ICAO24) AS Aircraft, sum(Km) AS Km
FROM BrusselsDepartures;

## Query 3: the flights over each country and the distance they fly there

Each flight is restricted to each country that it intersects, and the length of the restriction
is measured on the sphere, after projecting it back to WGS84 and casting it to a `tgeogpoint`.

In [ ]:
%%sql

DROP TABLE IF EXISTS CountryTraffic;

CREATE TABLE CountryTraffic USING DELTA AS
SELECT c.Name, COUNT(*) AS Flights,
  SUM(geoLength(tgeogpoint(geoTransform(atGeometry(f.Flight, c.Geom),
    4326)))) / 1000 AS Km
FROM FlightsEU f, Countries c
WHERE eIntersects(f.Flight, c.Geom)
GROUP BY c.Name;

SELECT Name, Flights, Km FROM CountryTraffic ORDER BY Km DESC LIMIT 10;

## Query 4: the number of aircraft in the air over Europe during the day

Each flight is restricted to the periods during which it flies, its velocity between 50 and
400 m/s, and to the envelope of Europe, and the temporal aggregate `tCount` counts the restricted
flights at each instant of the day. Spark computes a partial count on each executor and merges the
partial counts, as it merges the partial results of `COUNT`.

In [ ]:
%%sql

DROP TABLE IF EXISTS AirborneCount;

CREATE TABLE AirborneCount USING DELTA AS
WITH Europe(Env) AS (
  SELECT geoTransform(segmentize(geometryFromHexEWKB(ST_AsHEXEWKB(ST_SetSRID(
    ST_PolygonFromEnvelope(-35.58, 24.60, 44.83, 84.73), 4326))), 1), 3035) ),
Flying(Trip) AS (
  SELECT atGeometry(atTime(Flight,
    getTime(atSpan(Velocity, floatspanFromText('[50, 400]')))), Env)
  FROM FlightsEU, Europe
  WHERE atSpan(Velocity, floatspanFromText('[50, 400]')) IS NOT NULL )
SELECT tCount(Trip) AS Aircraft
FROM Flying
WHERE Trip IS NOT NULL;

SELECT numInstants(Aircraft) AS Instants, maxValue(Aircraft) AS Peak,
  startTimestamp(atValue(Aircraft, maxValue(Aircraft))) AS PeakTime,
  twAvg(Aircraft) AS Average
FROM AirborneCount;

SELECT h AS Hour, valueAtTimestamp(Aircraft, h) AS Aircraft
FROM AirborneCount
LATERAL VIEW explode(sequence(TIMESTAMP '2026-03-01 00:00:00',
  TIMESTAMP '2026-03-01 23:00:00', INTERVAL 1 HOUR)) e AS h
ORDER BY h;

## Query 5: the flights holding over London

Each flight is restricted to the periods during which it flies between 2,000 and 5,000 meters,
and then to the periods during which it lies within 15 km of one of the four holding fixes of
London Heathrow, given by `whenTrue(tDwithin(...))`. The flights are the temporal geography
points of `Flights`, so `tDwithin` measures the distance on the WGS 84 spheroid, and its answer
changes at each instant at which that distance crosses 15 km. The duration of the restriction is
the time that the flight spends there.

In [ ]:
%%sql

DROP TABLE IF EXISTS Holding;

CREATE TABLE Holding USING DELTA AS
WITH Fixes(Fix, Geog) AS (
  SELECT Fix, geographyFromHexEWKB(ST_AsHEXEWKB(
    ST_SetSRID(ST_Point(Lon, Lat), 4326)))
  FROM VALUES ('Bovingdon', -0.5500, 51.7260), ('Lambourne', 0.1520, 51.6460),
    ('Ockham', -0.4470, 51.3050), ('Biggin Hill', 0.0350, 51.3310)
    AS f(Fix, Lon, Lat) ),
Band(ICAO24, CallSign, Trip) AS (
  SELECT ICAO24, CallSign,
    atTime(Flight, getTime(atSpan(GeoAltitude, floatspanFromText('[2000, 5000]'))))
  FROM Flights
  WHERE atSpan(GeoAltitude, floatspanFromText('[2000, 5000]')) IS NOT NULL ),
Near(Fix, ICAO24, CallSign, Trip) AS (
  SELECT f.Fix, b.ICAO24, b.CallSign,
    atTime(b.Trip, whenTrue(tDwithin(b.Trip, f.Geog, 15000.0))) AS Trip
  FROM Band b, Fixes f
  WHERE eDwithin(b.Trip, f.Geog, 15000.0) )
SELECT Fix AS Fix, ICAO24 AS ICAO24, CallSign AS CallSign, Trip AS Trip
FROM Near
WHERE Trip IS NOT NULL;

SELECT Fix, count(*) AS Flights,
  count_if(duration(Trip) >= INTERVAL '10' MINUTE) AS Holding10,
  max(duration(Trip)) AS Longest
FROM Holding
GROUP BY Fix
ORDER BY Fix;

SELECT Fix, ICAO24, CallSign, duration(Trip) AS Holding,
  geoLength(Trip) / 1000 AS Km
FROM Holding
WHERE duration(Trip) >= INTERVAL '10' MINUTE
ORDER BY Holding DESC
LIMIT 10;

## The rows of the maps

The answers are written as CSV files into the folder `flights_queries` beside the notebook, the
input of `figures/flights_queries_maps.py`: each trajectory, route, and destination is written by
MEOS in HexEWKB and read by Sedona, which writes it in WKT, as it writes the boundary of each
country from the shapefile.

In [ ]:
import csv
import os

os.makedirs("flights_queries", exist_ok=True)
EXPORTS = {
    "q1_rows": "SELECT ICAO24, CallSign, geoLength(Trip) AS Length, "
               "ST_AsText(ST_GeomFromEWKB(unhex(asHexEWKB(trajectory(geoTransform(Trip, 4326)))))) "
               "AS WKT FROM MorningFlights",
    "q2_rows": "SELECT ICAO24, CallSign, Km, ST_AsText(ST_GeomFromEWKB(unhex(asHexEWKB(Route)))) "
               "AS WKT, ST_AsText(ST_GeomFromEWKB(unhex(asHexEWKB(Destination)))) AS Destination "
               "FROM BrusselsDepartures",
    "q3_rows": "SELECT c.Name, c.Flights, c.Km, ST_AsText(i.geometry) AS WKT "
               "FROM CountryTraffic c, CountriesInput i "
               "WHERE trim(TRAILING chr(0) FROM i.NAME) = c.Name ORDER BY c.Name",
    "q4_series": "SELECT asText(Aircraft) AS Aircraft FROM AirborneCount",
    "q5_rows": "SELECT Fix, ICAO24, CallSign, CAST(duration(Trip) AS STRING) AS Holding, "
               "geoLength(Trip) / 1000 AS Km, "
               "ST_AsText(ST_GeomFromEWKB(unhex(asHexEWKB(trajectory(Trip))))) AS WKT "
               "FROM Holding ORDER BY Fix, ICAO24, CallSign",
}
for name, query in EXPORTS.items():
    rows = spark.sql(query).collect()
    with open(f"flights_queries/{name}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(rows[0].__fields__)
        w.writerows(["" if v is None else v for v in r] for r in rows)
    print(name, len(rows), "rows")